In [1]:
import sys; sys.path.append("..")
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.data import load_adult
from src.models import build_models

os.makedirs("../results", exist_ok=True)
X_train, X_test, y_train, y_test = load_adult()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "f1": "f1"}

In [2]:
fold_rows = []
for name, model in build_models(X_train).items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    for i in range(cv.get_n_splits()):
        fold_rows.append({
            "model": name, "fold": i,
            "roc_auc": s["test_roc_auc"][i],
            "pr_auc": s["test_pr_auc"][i],
            "f1": s["test_f1"][i],
            "fit_time_s": s["fit_time"][i],
        })
    print("done:", name)

folds = pd.DataFrame(fold_rows)
folds.to_csv("../results/cv_folds_default_models.csv", index=False)

done: Logistic Regression (baseline)
done: Random Forest (bagging)
done: XGBoost (boosting)
done: Stacking (RF + XGB -> LR)


In [3]:
def fmt(col):
    g = folds.groupby("model", sort=False)[col]
    return g.mean().round(4).astype(str) + " ± " + g.std().round(4).astype(str)

summary = pd.DataFrame({
    "ROC-AUC": fmt("roc_auc"),
    "PR-AUC": fmt("pr_auc"),
    "F1": fmt("f1"),
    "fit time (s)": folds.groupby("model", sort=False)["fit_time_s"].mean().round(1),
})
summary.to_csv("../results/cv_summary_default_models.csv")
summary

,ROC-AUC,PR-AUC,F1,fit time (s)
model,,,,
Logistic Regression (baseline),0.9064 ± 0.0043,0.7653 ± 0.0123,0.6589 ± 0.011,0.5
Random Forest (bagging),0.891 ± 0.0058,0.7339 ± 0.012,0.6558 ± 0.0063,17.5
XGBoost (boosting),0.9242 ± 0.003,0.8191 ± 0.007,0.7042 ± 0.0099,1.0
Stacking (RF + XGB -> LR),0.9239 ± 0.0031,0.8185 ± 0.0071,0.6978 ± 0.0089,110.7
